In [1]:
import torch as pt


mols_test = pt.load('./data/mine/test_11499.pt')
print(len(mols_test))
mols_all = pt.load('./data/mine/mols_all.pt')
print(len(mols_all))

11499
2253216


In [2]:
# 统计词频
import numpy as np


mols_train = mols_all[:232826]


In [88]:
from torch.utils.data import DataLoader
from utils.data import SpecDataset
import numpy as np
from torch import from_numpy


def collate_fun_conv(batch):
    mzs_intens = []
    for mz, inten in batch:
        mz_inten = np.zeros(1000, dtype=np.float32)
        mz_inten[mz] = inten
        mzs_intens.append(mz_inten)
    return pt.tensor(np.array(mzs_intens), dtype=pt.float32)



dataset_lib = SpecDataset(mols_all)
dataset_test = SpecDataset(mols_test)
loader_lib = DataLoader(dataset_lib, batch_size=2048, shuffle=False,
                        num_workers=8, collate_fn=collate_fun_conv)
loader_test = DataLoader(dataset_test, batch_size=2048, shuffle=False,
                        num_workers=8, collate_fn=collate_fun_conv)
dataset_train = SpecDataset(dataset_lib, mapping=np.arange(232826))
loader_train = DataLoader(dataset_train, batch_size=256, shuffle=True, 
                            num_workers=8, collate_fn=collate_fun_conv)
num_batches = len(loader_train)

In [97]:
import torch.optim as optim

import torch.nn as nn
import torch.nn.functional as F


class SpecConv(nn.Module):
    def __init__(self, spec_len:int=1000):
        super(SpecConv, self).__init__()
        self.conv1 = nn.Conv1d(in_channels=1 , out_channels=2, kernel_size=3, stride=2, padding=1)
        self.conv2 = nn.Conv1d(in_channels=2, out_channels=4, kernel_size=3, stride=2, padding=1)
        self.conv3 = nn.Conv1d(in_channels=4, out_channels=8, kernel_size=3, stride=2, padding=1)
        self.conv = nn.Sequential(self.conv1, nn.ReLU(), 
                                  self.conv2, nn.ReLU(), 
                                  self.conv3)
        self.tconv1 = nn.ConvTranspose1d(in_channels=8, out_channels=4, kernel_size=3, stride=2, padding=1, output_padding=1)
        self.tconv2 = nn.ConvTranspose1d(in_channels=4, out_channels=2, kernel_size=3, stride=2, padding=1, output_padding=1)
        self.tconv3 = nn.ConvTranspose1d(in_channels=2, out_channels=1, kernel_size=3, stride=2, padding=1, output_padding=1)
        self.tconv = nn.Sequential(self.tconv1, nn.ReLU(), 
                                   self.tconv2, nn.ReLU(), 
                                   self.tconv3)

    def forward(self, mzs_intens, mode:str='train'):
        if mode == 'train': 
            return self.tconv(self.conv(mzs_intens))
        elif mode == 'emb': # emb模式下的masks只mask掉了padding 
            return (self.conv(mzs_intens)).sum(dim=1)
        else:
            raise ValueError('mode not exist')


gpu = 9
model = SpecConv().to(gpu)

epochs = 10
lr = 0.001
optimizer = optim.Adam(model.parameters(), lr=lr)

In [98]:
from tqdm import tqdm
from utils.tools import build_idx, evaluate, save_model


def gen_embeddings(model:nn.Module, loader:DataLoader, gpu:int):
    model.eval()
    embs = []
    with pt.no_grad():
        for mzs_intens in loader:
            data = mzs_intens.unsqueeze(1).to(gpu)
            emb = model(data, mode='emb').detach().cpu().numpy()
            embs.append(emb)
    pt.cuda.empty_cache()
    embs = np.concatenate(embs, axis=0)
    embs /= np.linalg.norm(embs, axis=1, keepdims=True)
    return embs


f = open('conv.txt', 'w')
model_name = 'conv'
max_metrics = {'expand': [0, 0], 'insilico': [0, 0]}
for epoch in range(epochs):
    print(f'==================================Train_epoch{epoch+1}======================================')
    model.train()
    train_loss = []
    for i, mzs_intens in enumerate(tqdm(loader_train, unit='batch')):
        data = mzs_intens.unsqueeze(1).to(gpu)
        optimizer.zero_grad()
        output = model(data)
        loss = ((output - data)**2).sum()
        train_loss.append(loss.item())
        loss.backward()
        optimizer.step()
        if (i+1) %300 ==0:
            loss = np.mean(train_loss)
            print(f'Total Loss: {loss}')
            train_loss = []
    
    print(f'===================================Test_epoch{epoch+1}======================================')
    f.write('\nTest_epoch%d\n' % (epoch+1))
    embeddings_lib = gen_embeddings(model, loader_lib, gpu)
    embeddings_test = gen_embeddings(model, loader_test, gpu)
    I_expand, _ = build_idx(embeddings_lib, embeddings_test, gpu)
    top1_expand, top10_expand = evaluate(mols_test, I_expand, mols_all, f, 'Expanded')
    # if top1_expand > max_metrics['expand'][0] and top10_expand > max_metrics['expand'][1]:
    #     max_metrics['expand'] = [top1_expand, top10_expand]
    #     save_model(model, model_name, epoch)
    I_insilico, _ = build_idx(embeddings_lib[:2146690], embeddings_test, gpu)
    top1_insilico, top10_insilico = evaluate(mols_test, I_insilico, mols_all, f, 'In-silico')
    # if top1_insilico > max_metrics['insilico'][0] and top10_insilico > max_metrics['insilico'][1]:
    #     max_metrics['insilico'] = [top1_insilico, top10_insilico]
    #     save_model(model, model_name, epoch)
    print(f'================================================================================================')
f.close()

==================================Train_epoch1======================================


  0%|          | 0/910 [00:00<?, ?batch/s]

 35%|███▍      | 317/910 [00:08<00:05, 104.98batch/s]

Total Loss: 10318.50270304362


 69%|██████▉   | 628/910 [00:10<00:01, 154.32batch/s]

...

100%|█████████▉| 909/910 [00:13<00:00, 101.93batch/s]

Total Loss: 139.9778621927897


100%|██████████| 910/910 [00:14<00:00, 60.98batch/s] 

===================================Test_epoch6======================================


KeyboardInterrupt: 